# Exercise 09 — The Capital Asset Pricing Model

MSc Finance · Investments · FHNW · Autumn 2026

Every exercise so far has taken expected returns as given. Exercise 05 fed them into an
optimizer and watched the weights explode; Exercise 06 estimated betas against an index
that was simply assumed to be the right one. Today we run the argument in the other
direction. Three companies, three investors, one risk-free asset — and the question is
not *which portfolio should I hold* but *what must expected returns be so that everyone
can hold what actually exists*.

There is no data file this week. The economy is fourteen numbers, and every result in
this notebook can be checked with a pen. That is the point: when the answer comes out
wrong you will know it is the reasoning, not the data.

**One change from the lecture.** Slide 16 set all pairwise correlations to zero so that
the arithmetic could be done at the whiteboard. Here $\rho = 0.20$ throughout. Nothing
in the theory changes. Almost every number does.

**How to work with this notebook.** This is a worked example: every cell is complete.
Each task states a question, the code under it answers it, and the commentary after the
code says what the result means and where people usually go wrong. Before you run a code
cell, read the task and decide what you expect to see. The cells are short enough for
that, and the habit is exactly what the exercise sheet next Monday rewards. The
**Your turn** block at the end asks three questions that need no new code: you change one
number at a time and read what moves.

Run **Runtime → Restart and run all** before you trust any number in here. Nothing in
this notebook draws random numbers, so everyone's output is identical to the last
decimal.

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from exercise_utils import FHNW, setup_style, save_results
setup_style()

### The economy

Three Swiss-listed companies, three investors, one risk-free asset in zero net supply.
All figures in CHF billion. This is slide 16, with one number changed.

The cell below is complete. Read it before you run it — the six lines after `RHO` are
the entire model.

In [ ]:
ASSETS = ["A", "B", "C"]

# --- the three companies ---------------------------------------------------
cap = pd.Series([250.0, 150.0, 100.0], index=ASSETS)   # market capitalisation
sd = pd.Series([0.20, 0.20, 0.20], index=ASSETS)       # volatility
w_m = cap / cap.sum()                                  # market portfolio weights

# --- the three investors ---------------------------------------------------
investors = pd.DataFrame(
    {"wealth": [200.0, 200.0, 100.0], "A": [5.0, 2.5, 1.25]},
    index=["I  pension fund", "II  private investor", "III  leveraged fund"])

rf = 0.02

# --- the one assumption we change against the lecture ----------------------
RHO = 0.20                                             # slide 16 used 0.0

corr = np.full((3, 3), RHO)
np.fill_diagonal(corr, 1.0)
Sigma = pd.DataFrame(np.outer(sd, sd) * corr, index=ASSETS, columns=ASSETS)

# The expected excess returns the lecture worked with. They belong to the rho = 0
# economy of slide 16. Whether they survive rho = 0.20 is Task 1.
mu_slide = pd.Series([0.05, 0.03, 0.02], index=ASSETS)

print("market weights\n" + (100 * w_m).round(1).to_string())
print("\ncovariance matrix\n" + Sigma.to_string())
print("\ntotal wealth", investors["wealth"].sum(), " total market cap", cap.sum())

## Task 1 — The lecture's numbers stop working

Slide 16 gave the three companies expected excess returns of 5 %, 3 % and 2 %, and
slide 18 showed that the three investors between them end up holding exactly
50 / 30 / 20 — the market portfolio. That was an equilibrium *for* $\rho = 0$.

Keep those three expected excess returns. Change nothing else except $\rho = 0.20$, and
ask what portfolio of risky assets an investor now wants.

Every mean-variance investor holds the tangency portfolio, and there is a closed form
for it — no optimizer needed:

$$w_{TP} \;\propto\; \Sigma^{-1}\bigl(E(R) - R_f\bigr), \qquad\text{then rescale so that }
\sum_i w_i = 1 .$$

Write it as a function, because you will want it again in Task 3. Do **not** invert
$\Sigma$ explicitly — `np.linalg.solve(Sigma, mu)` returns $\Sigma^{-1}\mu$ directly and
is both faster and better behaved.

Then put the tangency weights next to the market weights, and answer in one sentence:
if every investor in this economy wants the portfolio you have just computed, who holds
the rest of company C?

*Deliverable: the tangency weights against the market weights, as one printed table, and
that one sentence.*

In [ ]:
# tangency(mu, Sigma) -> weights proportional to inv(Sigma) @ mu, summing to 1
def tangency(mu, Sigma=Sigma):
    z = np.linalg.solve(Sigma, mu)
    return pd.Series(z / z.sum(), index=mu.index)


# the tangency portfolio implied by the lecture's expected excess returns
w_tp_slide = tangency(mu_slide)

comparison = pd.DataFrame({"market": 100 * w_m,
                           "wanted": 100 * w_tp_slide})
comparison["gap"] = comparison["wanted"] - comparison["market"]
print("percent of the risky portfolio\n" + comparison.round(1).to_string())

### What Task 1 shows

The tangency portfolio is **62.5 % A, 27.5 % B, 10.0 % C**. Every investor wants it, yet
C is 20 % of the market: half of C has no buyer at these prices, and A is oversubscribed
by 12.5 points. The market does not clear — slide 18 worked only for $\rho = 0$.

C suffers because its only claim on a portfolio was that its risk could be diversified
away. A higher $\rho$ weakens that claim, and the low-premium asset is dropped first.

The mistake to expect: reading the gap as "C is overpriced". Nothing is priced yet. The
shares are in fixed supply, so in Task 2 it is the *expected returns* that give way, not
the quantities.

## Task 2 — Reverse the map

The CAPM's move is to refuse the question Task 1 asked. Instead of computing weights
from expected returns, it takes the weights as observed — they are the market
capitalisations, which are public — and asks what expected returns would make them
optimal.

Three steps, in this order.

**Step 1: the price of risk.** Each investor $j$ puts a share $y_j^* = \dfrac{E(R_M) -
R_f}{A_j \sigma_M^2}$ of their wealth in the risky portfolio. Total risky demand must
equal total market capitalisation, and total wealth equals total market capitalisation
too, because the risk-free asset is borrowing between these three investors and nets to
zero. Impose that, and the market's aggregate risk aversion $\bar{A}$ falls out:

$$\frac{1}{\bar{A}} \;=\; \sum_j \frac{W_j / W}{A_j} .$$

**Step 2: the market risk premium.** $E(R_M) - R_f = \bar{A}\,\sigma_M^2$, with
$\sigma_M^2 = w_M' \Sigma\, w_M$.

**Step 3: every individual asset.** $\beta_i = \dfrac{\operatorname{Cov}(R_i, R_M)}{\sigma_M^2}$,
where the whole vector of covariances is one matrix product $\Sigma w_M$, and then
$E(R_i) - R_f = \beta_i\bigl(E(R_M) - R_f\bigr)$.

Build the equilibrium table — covariance with the market, beta, expected excess return,
expected return — and then run the three checks that tell you it is an equilibrium:
the value-weighted average beta, each investor's $y^*$, and whether the tangency
portfolio of your new expected returns is the market portfolio.

Finally, plot the economy twice: in $(\sigma, E(R))$ space with the capital allocation
line through M, and in $(\beta, E(R))$ space with the security market line. Same three
assets, two coordinate systems.

*Deliverable: one equilibrium table, three checks that pass, and the two-panel figure.*

In [ ]:
# aggregate risk aversion from the three investors (step 1)
w_wealth = investors["wealth"] / investors["wealth"].sum()
A_bar = 1.0 / (w_wealth / investors["A"]).sum()

# market variance, market risk premium (step 2)
var_m = w_m @ Sigma @ w_m
sd_m = np.sqrt(var_m)
prem_m = A_bar * var_m

# covariances with the market, betas, equilibrium excess returns (step 3)
cov_m = Sigma @ w_m
beta = cov_m / var_m
mu_eq = beta * prem_m

equilibrium = pd.DataFrame({"Cov(Ri,RM)": cov_m, "beta": beta,
                            "E(R)-Rf": 100 * mu_eq,
                            "E(R)": 100 * (rf + mu_eq)})
print(equilibrium.round(4).to_string())
print(f"\nA_bar = {A_bar:.4f}   sigma_M = {100 * sd_m:.4f}%   "
      f"E(RM)-Rf = {100 * prem_m:.4f}%   SR_M = {prem_m / sd_m:.4f}")

In [ ]:
# check 1 — the value-weighted average beta
print("value-weighted beta   ", round(w_m @ beta, 6))

# check 2 — each investor's optimal risky share
y_star = prem_m / (investors["A"] * var_m)
print("\ny* per investor\n" + (100 * y_star).round(1).to_string())

# check 3 — the tangency portfolio of the equilibrium expected returns
print("\ntangency vs market, percent\n" +
      pd.DataFrame({"tangency": 100 * tangency(mu_eq),
                    "market": 100 * w_m}).round(2).to_string())

# and the alphas, which is the same statement said differently
alpha_eq = mu_eq - beta * prem_m
print("\nalphas, percent\n" + (100 * alpha_eq).round(6).to_string())

In [ ]:
fig1, axes = plt.subplots(1, 2, figsize=(11.0, 4.6))

# --- left panel: the portfolio view ---------------------------------------
ax = axes[0]
x = np.linspace(0, 0.26, 50)
# plot the capital allocation line -- from Rf, with the market's Sharpe ratio
ax.plot(100 * x, 100 * (rf + (prem_m / sd_m) * x), color=FHNW["navy"],
        lw=2.0, label="CAL through M")
ax.scatter(100 * sd_m, 100 * (rf + prem_m), s=90, color=FHNW["navy"],
           zorder=3, label="Market portfolio")
ax.annotate("M", (100 * sd_m, 100 * (rf + prem_m)),
            textcoords="offset points", xytext=(7, -12))
ax.scatter(100 * sd, 100 * (rf + mu_eq), s=70, color=FHNW["blue"],
           zorder=3, label="Individual assets")
for i in ASSETS:
    ax.annotate(i, (100 * sd[i], 100 * (rf + mu_eq[i])),
                textcoords="offset points", xytext=(7, -3))
ax.scatter(0, 100 * rf, s=50, color=FHNW["green"], zorder=3)
ax.annotate("$R_f$", (0, 100 * rf), textcoords="offset points", xytext=(7, -3))
ax.set_xlabel("Standard deviation, % p.a.")
ax.set_ylabel("Expected return, % p.a.")
ax.legend(loc="lower right", fontsize=9)

# --- right panel: the pricing view ----------------------------------------
ax = axes[1]
b = np.linspace(0, 1.4, 50)
# plot the security market line -- from Rf through the market at beta = 1
ax.plot(b, 100 * (rf + b * prem_m), color=FHNW["navy"], lw=2.0, label="SML")
ax.scatter(1.0, 100 * (rf + prem_m), s=90, color=FHNW["navy"], zorder=3,
           label="Market portfolio")
ax.scatter(beta, 100 * (rf + mu_eq), s=70, color=FHNW["blue"], zorder=3,
           label="Individual assets")
for i in ASSETS:
    ax.annotate(i, (beta[i], 100 * (rf + mu_eq[i])),
                textcoords="offset points", xytext=(7, -3))
ax.set_xlabel(r"$\beta$")
ax.set_ylabel("Expected return, % p.a.")
ax.legend(loc="lower right", fontsize=9)

fig1.tight_layout()

### What Task 2 shows

Aggregate risk aversion is $\bar{A} = 2.5$, the wealth-weighted *harmonic* mean: demands
add up, and demand is proportional to $1/A_j$. With $\sigma_M = 14.20\ \%$ the market risk
premium is $\bar{A}\sigma_M^2 = 5.04\ \%$:

| | $\beta_i$ | $E(R_i) - R_f$ |
|---|---|---|
| A | 1.190 | 6.0 % |
| B | 0.873 | 4.4 % |
| C | 0.714 | 3.6 % |

All three checks pass. The value-weighted beta is 1; the investors hold 50 / 100 / 200 %
in risky assets, so the pension fund lends exactly the CHF 100 bn the leveraged fund
borrows; and the tangency portfolio is 50 / 30 / 20. We did not verify the CAPM — we
solved for the expected returns that make it true.

Against the lecture's $\rho = 0$, the premium rose from 3.8 % to 5.04 % with $\bar{A}$
unchanged: correlation raised the quantity of risk ($\sigma_M^2$ from 0.0152 to 0.0202),
not its price. The betas compressed from 1.32 / 0.79 / 0.53 to 1.19 / 0.87 / 0.71.

In the figure the three assets sit *below* the CAL (Sharpe ratios 0.30, 0.22 and 0.18
against 0.355) and exactly *on* the SML. Both are correct: only systematic risk is priced.

The mistake to expect: the wealth-weighted *arithmetic* mean, 3.25, gives a premium of
6.55 %. Check 2 catches it: the investors would hold CHF 650 bn of risky assets in a
market worth CHF 500 bn.

## Task 3 — Good news about C

Start from the equilibrium of Task 2. Analysts at one firm learn something good about
company C: its expected excess return is now **7.6 %**, four percentage points above its
equilibrium value of 3.6 % in Task 2. Prices have not moved, so the market capitalisations are
still 250 / 150 / 100. Volatilities and correlations have not moved either, so no beta
changes.

Work out what that does to the economy.

1. The market portfolio contains C, so the market risk premium moves too. Recompute it,
   then the SML prediction and the alpha for each of the three assets. Check what the
   three alphas sum to when weighted by market capitalisation.
2. Recompute the tangency portfolio with `tangency()` from Task 1, and compare its
   Sharpe ratio with the market's.
3. Before the news, C traded at 100 with an expected payoff of 105.60, its equilibrium
   expected return of 5.6 %. The news revises the expected payoff to 109.60. Prices adjust
   until the market is back in equilibrium and every alpha is zero again. Nothing about
   risk or risk aversion has changed. At what price does C trade then?

Then redraw the two-panel figure of Task 2 with the new numbers, and answer: why do A
and B have non-zero alphas when no news arrived about either of them?

*Deliverable: the alpha table, the new tangency portfolio with both Sharpe ratios, the
restored price, and the figure.*

In [ ]:
# the revised expected excess returns, and the market premium they imply
mu_news = mu_eq.copy()
mu_news["C"] = 0.076
prem_news = w_m @ mu_news

# the SML prediction and the alpha for each asset -- betas are unchanged
sml = beta * prem_news
alpha = mu_news - sml

news = pd.DataFrame({"E(R)-Rf": 100 * mu_news, "beta": beta,
                     "SML": 100 * sml, "alpha": 100 * alpha})
print(news.round(4).to_string())
print(f"\nmarket premium {100 * prem_m:.2f}% -> {100 * prem_news:.2f}%")
print(f"value-weighted alpha  {w_m @ alpha:.2e}")

In [ ]:
# the tangency portfolio of the revised expected returns
w_tp = tangency(mu_news)
sd_tp = np.sqrt(w_tp @ Sigma @ w_tp)
sr_tp = (w_tp @ mu_news) / sd_tp
sr_m_news = prem_news / sd_m

print("tangency vs market, percent\n" +
      pd.DataFrame({"tangency": 100 * w_tp,
                    "market": 100 * w_m}).round(2).to_string())
print(f"\nSR of the market   {sr_m_news:.4f}"
      f"\nSR of the tangency {sr_tp:.4f}"
      f"\ninformation ratio  {np.sqrt(sr_tp**2 - sr_m_news**2):.4f}")

# the price once the market is back in equilibrium: risk and risk aversion are unchanged,
# so the premium returns to A_bar * var_m and C's required return to its old value
payoff_new = 109.60
required = rf + beta["C"] * prem_m
price_new = payoff_new / (1 + required)
print(f"\nC: required return {100 * required:.4f}%   "
      f"price 100.00 -> {price_new:.2f}  ({100 * (price_new / 100 - 1):+.2f}%)")

# check: reprice C, update the market weights, and recompute every alpha
cap_new = cap.copy()
cap_new["C"] *= price_new / 100
w_new = cap_new / cap_new.sum()
mu_after = mu_news.copy()
mu_after["C"] = payoff_new / price_new - 1 - rf
alpha_after = mu_after - beta * (w_new @ mu_after)
print(f"market premium after the price jump {100 * (w_new @ mu_after):.2f}%")
print("alphas after the price jump, percent\n" + (100 * alpha_after).round(3).to_string())

In [ ]:
fig2, axes = plt.subplots(1, 2, figsize=(11.0, 4.6))

ax = axes[0]
x = np.linspace(0, 0.22, 50)
# two lines now -- through M, and the steeper one through the tangency
ax.plot(100 * x, 100 * (rf + sr_m_news * x), color=FHNW["navy"], lw=2.0,
        label=f"through M, SR = {sr_m_news:.3f}")
ax.plot(100 * x, 100 * (rf + sr_tp * x), color=FHNW["blue"], lw=2.0,
        ls="--", label=f"through TP, SR = {sr_tp:.3f}")
ax.scatter([100 * sd_m, 100 * sd_tp],
           [100 * (rf + prem_news), 100 * (rf + w_tp @ mu_news)],
           s=90, color=[FHNW["navy"], FHNW["blue"]], zorder=3)
ax.annotate("M", (100 * sd_m, 100 * (rf + prem_news)),
            textcoords="offset points", xytext=(8, -14))
ax.annotate("TP", (100 * sd_tp, 100 * (rf + w_tp @ mu_news)),
            textcoords="offset points", xytext=(-26, 4))
ax.set_xlabel("Standard deviation, % p.a.")
ax.set_ylabel("Expected return, % p.a.")
ax.legend(loc="lower right", fontsize=9)

ax = axes[1]
b = np.linspace(0, 1.6, 50)
# the SML again -- same intercept, a different slope
ax.plot(b, 100 * (rf + b * prem_news), color=FHNW["navy"], lw=2.0, label="SML")
for i in ASSETS:
    ax.plot([beta[i], beta[i]], [100 * (rf + sml[i]), 100 * (rf + mu_news[i])],
            color=FHNW["red"], lw=1.4, ls=":")
ax.scatter(beta, 100 * (rf + mu_news), s=70, color=FHNW["blue"], zorder=3,
           label="Individual assets")
ax.scatter(1.0, 100 * (rf + prem_news), s=90, color=FHNW["navy"], zorder=3,
           label="Market portfolio")
for i in ASSETS:
    ax.annotate(f"{i}: $\\alpha$ = {100 * alpha[i]:+.2f}%",
                (beta[i], 100 * (rf + mu_news[i])),
                textcoords="offset points", xytext=(7, -3), fontsize=9)
ax.set_xlabel(r"$\beta$")
ax.set_ylabel("Expected return, % p.a.")
ax.legend(loc="lower right", fontsize=9)

fig2.tight_layout()

### What Task 3 shows

C is a fifth of the market, so its four-point news lifts the market risk premium from
5.04 % to **5.84 %**. The betas are unchanged, but the line they are multiplied by rotated:

| | $\beta_i$ | SML requires | $\alpha_i$ |
|---|---|---|---|
| A | 1.190 | 6.95 % | **−0.95 %** |
| B | 0.873 | 5.10 % | **−0.70 %** |
| C | 0.714 | 4.17 % | **+3.43 %** |

**A and B were repriced without being touched.** Alpha is a distance from a line, not a
property of an asset. Weighted by market capitalisation the alphas sum to zero — an
identity, because the market cannot have an alpha against itself. Anybody's positive
alpha is somebody else's negative one.

The tangency portfolio is now 33.3 / 17.8 / 48.9 %, with a Sharpe ratio of 0.457 against
0.411 for the index: indexing is no longer optimal for whoever knows the news.

Nothing about risk has changed, so once C is repriced the premium falls back to 5.04 %
and C's required return to 5.60 %. The price jumps to **103.79**, and every alpha is zero
again — A's and B's included, although their prices never moved. The mispricing is earned
once, as a 3.8 % price jump by whoever traded first, not as 3.43 % a year. Who sold? By
the model's own assumptions, nobody: the Grossman–Stiglitz problem of Lecture 12.

Two mistakes to expect: holding the premium at 5.04 % when computing the alphas (then
$\alpha_C = +4.00\ \%$ and A and B show nothing), and discounting the 109.60 at the
post-news 6.17 %, which gives 103.23 — a price at which C still has an alpha of about
0.5 %.

## Export

Bundle the figures and the tables, in case you want them for your own notes.

In [ ]:
# collect the figures and tables into the two dicts save_results expects
figures = {"equilibrium": fig1, "after_the_news": fig2}
tables = {"task1": comparison, "equilibrium": equilibrium, "news": news}

save_results(figures=figures, tables=tables, name="ex09")

## Your turn

Three questions, no new code. Each one changes at most one number in a cell you have
already run, and asks you to say what will move before you look.

Work in your own copy (File, then Save a copy in Drive), and put every number back when
you are done, so that the commentary above still matches your output. Write your answers
down, two or three sentences each. We go through them in the exercise block on
Monday 16 November, straight after the exercise sheet.

### 1 · Predict

The pension fund becomes twice as risk-averse: its $A$ goes from 5 to 10. Nothing else
in the economy changes.

Before you run anything, write down for each of these five results whether it goes up,
goes down or stays the same, with half a sentence on why:

- aggregate risk aversion $\bar{A}$
- the market risk premium $E(R_M) - R_f$
- the beta of company C
- the weight of C in the tangency portfolio
- the share $y^*$ that the leveraged fund (investor III) puts into risky assets

Then change the pension fund's `5.0` to `10.0` in the `"A"` list of the economy cell, run
**Runtime → Restart and run all**, and check your five answers against the output of
Task 2. Ignore Task 3 in this run: its 7.6 % was set for the old equilibrium. Set the
number back to `5.0` when you are done.

### 2 · Change one input

In Task 3 the good news was about C, a fifth of the market. Give exactly the same news to
A instead, which is half of the market. In the first code cell of Task 3, replace

    mu_news["C"] = 0.076

with

    mu_news["A"] = 0.100

which puts A four percentage points above its equilibrium 6.0 %. Run that cell and read
the alpha table.

Explain in two sentences why A's alpha is less than half of the alpha C received for the
same news, and why the two companies nobody learned anything about now have larger
negative alphas than A and B had in Task 3. Put the original line back and run the Task 3
cells again when you are done.

### 3 · Interpret

A private investor looks at the left panel of the Task 2 figure. Company C has a Sharpe
ratio of 0.18, against 0.30 for A and 0.355 for the market, and he concludes:

*"C is clearly the worst of the three. I'll hold A and B and leave C out."*

Use both panels of the figure to explain, in three or four sentences, what is wrong with
this conclusion and what it costs him.